# load dataset

In [1]:
import os
import numpy as np
import pandas as pd

# Show files (optional)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/hashes.txt
/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv
/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/database.sqlite


# Load reviews CSV

In [2]:
df = pd.read_csv('/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv')
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


# Keep only the text column and drop missing

In [3]:
df = df[['Text']].dropna().reset_index(drop=True)
df

,Text
0,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...
2,This is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...
...,...
568449,Great for sesame chicken..this is a good if no...
568450,I'm disappointed with the flavor. The chocolat...
568451,"These stars are small, so you can give 10-15 o..."
568452,These are the BEST treats for training and rew...


# use a smaller subset for speed

In [4]:
df_small = df.head(100)  # adjust if you want more/less
df_small

,Text
0,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...
2,This is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...
...,...
95,I've been very pleased with the Natural Balanc...
96,My 1-1/2 year old basenji/jack russell mix lov...
97,Our pup has experienced allergies in forms of ...
98,My English Bulldog had skin allergies the summ...


# Install & load spaCy

In [5]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")

# POS tagging with spaCy

In [7]:
# Take a single example
sample_text = df_small.loc[0, 'Text']
doc = nlp(sample_text)
doc

I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.

In [8]:
print("=== POS TAGGING EXAMPLE ===")
for token in doc:
    print(token.text, token.pos_, token.tag_)

=== POS TAGGING EXAMPLE ===
I PRON PRP
have AUX VBP
bought VERB VBN
several ADJ JJ
of ADP IN
the DET DT
Vitality PROPN NNP
canned VERB VBN
dog NOUN NN
food NOUN NN
products NOUN NNS
and CCONJ CC
have AUX VBP
found VERB VBN
them PRON PRP
all PRON DT
to PART TO
be AUX VB
of ADP IN
good ADJ JJ
quality NOUN NN
. PUNCT .
The DET DT
product NOUN NN
looks VERB VBZ
more ADV RBR
like ADP IN
a DET DT
stew NOUN NN
than ADP IN
a DET DT
processed VERB VBN
meat NOUN NN
and CCONJ CC
it PRON PRP
smells VERB VBZ
better ADV RBR
. PUNCT .
My PRON PRP$
Labrador PROPN NNP
is AUX VBZ
finicky ADJ JJ
and CCONJ CC
she PRON PRP
appreciates VERB VBZ
this DET DT
product NOUN NN
better ADV RBR
than ADP IN
  SPACE _SP
most ADJ JJS
. PUNCT .


# POS tagging for first N rows (store results)

In [9]:
pos_results = []

for i, text in df_small['Text'].items():
    doc = nlp(text)
    pos_tags = [(token.text, token.pos_) for token in doc]
    pos_results.append(pos_tags)

# Example: show POS tags for first review

In [10]:
print("\n=== POS TAGS FOR FIRST REVIEW ===")
print(pos_results[0])


=== POS TAGS FOR FIRST REVIEW ===
[('I', 'PRON'), ('have', 'AUX'), ('bought', 'VERB'), ('several', 'ADJ'), ('of', 'ADP'), ('the', 'DET'), ('Vitality', 'PROPN'), ('canned', 'VERB'), ('dog', 'NOUN'), ('food', 'NOUN'), ('products', 'NOUN'), ('and', 'CCONJ'), ('have', 'AUX'), ('found', 'VERB'), ('them', 'PRON'), ('all', 'PRON'), ('to', 'PART'), ('be', 'AUX'), ('of', 'ADP'), ('good', 'ADJ'), ('quality', 'NOUN'), ('.', 'PUNCT'), ('The', 'DET'), ('product', 'NOUN'), ('looks', 'VERB'), ('more', 'ADV'), ('like', 'ADP'), ('a', 'DET'), ('stew', 'NOUN'), ('than', 'ADP'), ('a', 'DET'), ('processed', 'VERB'), ('meat', 'NOUN'), ('and', 'CCONJ'), ('it', 'PRON'), ('smells', 'VERB'), ('better', 'ADV'), ('.', 'PUNCT'), ('My', 'PRON'), ('Labrador', 'PROPN'), ('is', 'AUX'), ('finicky', 'ADJ'), ('and', 'CCONJ'), ('she', 'PRON'), ('appreciates', 'VERB'), ('this', 'DET'), ('product', 'NOUN'), ('better', 'ADV'), ('than', 'ADP'), (' ', 'SPACE'), ('most', 'ADJ'), ('.', 'PUNCT')]


# NER for first N rows (store results)

In [11]:
ner_results = []

for i, text in df_small['Text'].items():
    doc = nlp(text)
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    ner_results.append(ents)

In [12]:
ner_results

[[('Vitality', 'ORG'), ('My Labrador', 'PERSON')],
 [('Jumbo Salted', 'ORG'), ('Jumbo', 'WORK_OF_ART')],
 [('around a few centuries', 'DATE'),
  ('pillowy citrus gelatin', 'ORG'),
  ('Filberts', 'GPE'),
  ("C.S. Lewis'", 'ORG'),
  ('The Lion, The Witch', 'WORK_OF_ART'),
  ('Edmund', 'PRODUCT'),
  ('Sisters', 'PERSON'),
  ('Witch', 'ORG')],
 [('Robitussin', 'GPE'), ('the Root Beer Extract I', 'LAW')],
 [('Delivery', 'PERSON')],
 [('five pound', 'QUANTITY'), ('only two weeks', 'DATE')],
 [("Fralinger's", 'ORG')],
 [],
 [('Wheatgrass', 'PERSON')],
 [],
 [('Tequila Picante Gourmet de Inclan', 'ORG')],
 [('One', 'CARDINAL')],
 [('Felidae Platinum', 'ORG'),
  ('more than two years', 'DATE'),
  ('first', 'ORDINAL')],
 [('Twizzlers', 'ORG')],
 [('The Strawberry Twizzlers', 'ORG'),
  ('Six pounds', 'QUANTITY'),
  ('I.', 'ORG')],
 [('six pounds', 'MONEY'), ('six', 'CARDINAL')],
 [],
 [('Twizzler', 'ORG')],
 [('Strawberry', 'PERSON'),
  ('Lancaster Pennsylvania', 'ORG'),
  ('Y & S Candies, Inc.',

# Example: show entities for first review

In [13]:
print("\n=== ENTITIES FOR FIRST REVIEW ===")
print(ner_results[0])


=== ENTITIES FOR FIRST REVIEW ===
[('Vitality', 'ORG'), ('My Labrador', 'PERSON')]


# Bag-of-Words (BoW)

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

In [15]:
# Use the same df_small texts
texts = df_small['Text'].tolist()

In [16]:
# Create BoW model
vectorizer = CountVectorizer(
    max_features=1000,      # limit vocab size
    stop_words='english'    # remove English stopwords
)

X_bow = vectorizer.fit_transform(texts)

In [17]:
# Vocabulary
vocab = vectorizer.get_feature_names_out()
print("\n=== BOW VOCAB SAMPLE ===")
print(vocab[:30])


=== BOW VOCAB SAMPLE ===
['10' '1300watt' '16' '1845' '19' '1998' '20' '200' '2x' '30' '370' '45'
 '50' '90' 'abby' 'abdominal' 'able' 'actually' 'add' 'advertised'
 'aftertaste' 'ages' 'ago' 'allergic' 'allergies' 'allergy' 'amazing'
 'amazon' 'amounts' 'apple']


In [18]:
# Show BoW matrix shape
print("\n=== BOW MATRIX SHAPE ===")
print(X_bow.shape)


=== BOW MATRIX SHAPE ===
(100, 1000)


In [19]:
# Convert first row to dense and show non-zero entries
first_row = X_bow[0].toarray().flatten()
non_zero_indices = np.where(first_row > 0)[0]

In [20]:
print("\n=== NON-ZERO WORDS IN FIRST REVIEW ===")
for idx in non_zero_indices:
    print(vocab[idx], first_row[idx])


=== NON-ZERO WORDS IN FIRST REVIEW ===
better 2
bought 1
dog 1
finicky 1
food 1
good 1
labrador 1
like 1
looks 1
meat 1
processed 1
product 2
products 1
quality 1
smells 1


# Summary prints

In [21]:
print("\n=== SUMMARY ===")
print("Number of texts used:", len(df_small))
print("POS results example:", pos_results[0][:10])
print("NER results example:", ner_results[0])


=== SUMMARY ===
Number of texts used: 100
POS results example: [('I', 'PRON'), ('have', 'AUX'), ('bought', 'VERB'), ('several', 'ADJ'), ('of', 'ADP'), ('the', 'DET'), ('Vitality', 'PROPN'), ('canned', 'VERB'), ('dog', 'NOUN'), ('food', 'NOUN')]
NER results example: [('Vitality', 'ORG'), ('My Labrador', 'PERSON')]
